# M2.2 Laplace and Poisson by Relaxation

Book pointer: chapter 6, sections 6.1 (Laplace and Poisson equations), 6.2 (boundary conditions), 6.3 (finite difference methods), 6.4 (implicit methods).

## 1. Recall

1. Derive Poisson's equation from $\mathbf{E} = -\nabla V$ and Gauss' law. When does it reduce to Laplace's equation?
2. Dirichlet versus Neumann boundary condition: what is fixed in each, and which one does a conductor at a known potential give?
3. Write the 2D five-point stencil for $V_{i,j}$. Where does the $h^2\rho/\epsilon_0$ term come from?
4. In words: why does repeatedly replacing each interior value by the average of its neighbours converge to the solution of Laplace's equation?
5. State the uniqueness theorem. What does it tell you about two different methods that both satisfy the same boundary conditions?

## 2. Principle to algorithm

Laplace's equation says each point's potential is the average of its surroundings. On a grid that is literally the five-point stencil, and it suggests the algorithm: start from any guess, sweep the grid replacing every free node by the stencil average, and repeat until nothing changes. Jacobi uses only old values during a sweep. Gauss-Seidel uses new values as soon as they exist, which roughly halves the sweeps. Over-relaxation (SOR) extrapolates each update past the Gauss-Seidel value by a factor $\omega$ between 1 and 2 and cuts the sweep count from order $N^2$ to order $N$ for an $N\times N$ grid.

The alternative is to see the stencil as one linear equation per free node and solve all of them at once. The matrix has five non-zeros per row, so a sparse direct solver handles grids of a few hundred thousand nodes in seconds. Both approaches live in `emlib.relax`; fixed nodes are marked with a boolean mask and keep their values.

### Animation

The potential surface of the grounded-box problem above, starting flat and relaxing under Gauss-Seidel sweeps. Information enters from the 1 V edge and diffuses inward one cell per sweep, which is why the sweep count scales with the grid size squared.

In [ ]:
from emlib import show
show.animation("m2_electrostatics", "RelaxationSurface")

## 3. Build

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from emlib import grids, fields, relax
from emlib.fields import EPS0, K

plt.rcParams["figure.figsize"] = (6, 5)

In [ ]:
# Problem: grounded box, a plate at V = 1 along the top edge, and a grounded square conductor inside.
def setup(n):
    V0 = np.zeros((n, n))
    fixed = np.zeros((n, n), dtype=bool)
    V0[:, -1] = 1.0                             # top edge (y = max) at 1 V
    fixed[0, :] = fixed[-1, :] = fixed[:, 0] = fixed[:, -1] = True
    a, b = int(0.35 * n), int(0.55 * n)         # embedded grounded block
    V0[a:b, a:b] = 0.0
    fixed[a:b, a:b] = True
    return V0, fixed

n = 61
V0, fixed = setup(n)
h = 1.0 / (n - 1)

V_j, hist_j = relax.solve_relax(V0, fixed, method="jacobi", tol=1e-6)
V_gs, hist_gs = relax.solve_relax(V0, fixed, method="gauss-seidel", tol=1e-6)
V_sor, hist_sor = relax.solve_relax(V0, fixed, omega=relax.sor_omega(n, n), tol=1e-6)
V_sp = relax.solve_sparse(V0, fixed)

print(f"sweeps to 1e-6:  Jacobi {len(hist_j)}   Gauss-Seidel {len(hist_gs)}   SOR {len(hist_sor)}  (omega = {relax.sor_omega(n, n):.3f})")
print(f"max |SOR - sparse| = {np.abs(V_sor - V_sp).max():.2e}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
x = np.linspace(0, 1, n)
X, Y = np.meshgrid(x, x, indexing="ij")
cs = axes[0].contourf(X, Y, V_sp, levels=21, cmap="viridis")
axes[0].contour(X, Y, V_sp, levels=11, colors="w", linewidths=0.5)
plt.colorbar(cs, ax=axes[0], label="V")
E = -fields.gradient(V_sp, h, h)
st = 3
axes[0].quiver(X[::st, ::st], Y[::st, ::st], E[::st, ::st, 0], E[::st, ::st, 1], color="w", alpha=0.7)
axes[0].set_aspect("equal"); axes[0].set_title("potential and field, sparse direct solve")

for hst, lbl in ((hist_j, "Jacobi"), (hist_gs, "Gauss-Seidel"), (hist_sor, "SOR")):
    axes[1].semilogy(hst, label=lbl)
axes[1].set_xlabel("sweep"); axes[1].set_ylabel("max change per sweep"); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.show()

### Exercise: write the Jacobi sweep yourself

Replace the body of `my_jacobi_step` with a vectorised NumPy version of the five-point average (slices, no Python loops), leaving fixed nodes untouched. The check compares one step of yours to one step from the library on the same input.

In [ ]:
def my_jacobi_step(V, fixed, src):
    """One Jacobi step. src is h^2 rho / eps0. Returns the new array."""
    Vn, _ = relax.jacobi_step(V, fixed, src)    # replace with your own slice-based average
    return Vn


Vt = np.random.default_rng(1).random((n, n)); Vt[fixed] = V0[fixed]
src = np.zeros((n, n))
ref, _ = relax.jacobi_step(Vt, fixed, src)
assert np.allclose(my_jacobi_step(Vt, fixed, src), ref)
print("jacobi step check passed")

## 4. Verify

Two closed-form checks.

Scaling of sweeps with grid size. For Gauss-Seidel the number of sweeps to a fixed tolerance grows like $N^2$, for SOR like $N$. Measure the slopes.

Poisson with a known answer. Uniform $\rho$ between two grounded plates, with the side edges fixed to the exact parabola $V = \rho\,x(L-x)/2\epsilon_0$, so that the whole grid must reproduce that parabola.

In [ ]:
sizes = np.array([21, 31, 41, 61, 81])
sw_gs, sw_sor = [], []
for m in sizes:
    V0m, fm = setup(m)
    sw_gs.append(len(relax.solve_relax(V0m, fm, tol=1e-6)[1]))
    sw_sor.append(len(relax.solve_relax(V0m, fm, omega=relax.sor_omega(m, m), tol=1e-6)[1]))
p_gs = np.polyfit(np.log(sizes), np.log(sw_gs), 1)[0]
p_sor = np.polyfit(np.log(sizes), np.log(sw_sor), 1)[0]
plt.loglog(sizes, sw_gs, "o-", label=f"Gauss-Seidel, slope {p_gs:.2f}")
plt.loglog(sizes, sw_sor, "s-", label=f"SOR, slope {p_sor:.2f}")
plt.xlabel("grid points per side N"); plt.ylabel("sweeps to 1e-6"); plt.legend(); plt.grid(True, which="both", alpha=0.3); plt.show()
assert p_gs > 1.6 and p_sor < 1.4

In [ ]:
m, rho0 = 81, 1e-9
xs = np.linspace(0, 1, m)
hm = xs[1] - xs[0]
exact = rho0 * xs * (1 - xs) / (2 * EPS0)
V0p = np.zeros((m, m)); V0p[:, 0] = V0p[:, -1] = exact
fixedp = np.zeros((m, m), dtype=bool)
Vp = relax.solve_sparse(V0p, fixedp, rho=np.full((m, m), rho0), h=hm)
err = np.abs(Vp - exact[:, None]).max() / exact.max()
print(f"max relative error of the Poisson solution: {err:.2e}")
assert err < 1e-8

## 5. Explore and rebuild

Explore. Put a charged conductor (fixed at $V_0$) inside a grounded box and shrink the box around it. Plot the potential along a line from the conductor to the wall for box sizes from twice the conductor width to ten times. How large must the box be before the solution near the conductor stops changing? Relate this to the choice of outer boundary in every relaxation problem.

Rebuild. Close this notebook and write a Gauss-Seidel solver with a fixed-node mask from a blank editor, then the sparse assembly.